# AlignVid object-addition pilot: generation on Kaggle

Runs `gen/run_pilot.py` from the pilot package (uploaded as a private Kaggle dataset `alignvid-pilot-package`).
Notebook settings: **Accelerator = GPU T4 x2**, **Internet = On**, Persistence = off.

Heavy files (code, weights ≈ 40 GB) live in `/kaggle/tmp/pilot`, which is not saved. Finished videos, sidecars and logs are copied to `/kaggle/working` as they appear; that folder is saved as the notebook output.

Session plan (one Kaggle run ≤ 12 h; a 6 s video takes ≈ 65 min on the T4 in fp16):
1. **Session A**: `ARMS = ["vanilla"]` → length probe (6.0 → 3.0 → 2.0 s on sample_240, the successful probe *is* the first vanilla video) + the other four vanilla videos.
2. **Session B**: `ARMS = ["alignvid"]`, with session A's output attached as an input (*Add Input → Your Work → this notebook*) so the chosen length and finished videos are restored.
3. **Session C** (if quota remains): `ARMS = ["noedit"]`, with A and B attached.

Run each session with **Save Version → Save & Run All (Commit)** so it does not depend on the browser tab. Cells 1–4 take ≈ 10 min (venv, weights); cell 6 does the work.


In [1]:
# 0. settings for this session
ARMS = ["vanilla"]           # ["vanilla"] session A, ["alignvid"] session B, ["noedit"] session C
DO_DRYRUN = False            # True only to re-check a new machine (2 s probe, ≈ 30 min in fp16)
TRANSFORMER_DTYPE = "fp16"   # protocol §9 (2026-11-22): fp16 with fp32 residual on the T4; use "bf16" on Ampere or newer GPUs


In [2]:
# 1. unpack the package; restore results from previous versions of this notebook, if attached as inputs
import glob, os, shutil, subprocess, zipfile
ROOT = '/kaggle/tmp/pilot/pilot_package'
zips = glob.glob('/kaggle/input/**/pilot_package.zip', recursive=True)
dirs = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/gen/run_pilot.py', recursive=True)]
if not os.path.exists(ROOT):
    os.makedirs('/kaggle/tmp/pilot', exist_ok=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall('/kaggle/tmp/pilot')
    elif dirs:
        shutil.copytree(dirs[0], ROOT)
    else:
        raise SystemExit('package not found under /kaggle/input: attach the dataset (Add Input)')
for prev in sorted(glob.glob('/kaggle/input/*/outputs') + glob.glob('/kaggle/input/*/logs')):
    shutil.copytree(prev, os.path.join(ROOT, os.path.basename(prev)), dirs_exist_ok=True)
    print('restored', prev)
os.chdir(ROOT); print(open('VERSION.json').read())
os.environ.update(PILOT_HOST='kaggle', MPLBACKEND='Agg', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
                  PILOT_TRANSFORMER_DTYPE=TRANSFORMER_DTYPE, HF_HUB_DISABLE_XET='1')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout.splitlines()[:10])
print(subprocess.run(['df', '-h', '/kaggle/tmp', '/kaggle/working'], capture_output=True, text=True).stdout)
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)


{
 "packaged": "2026-11-22T09:43:36",
 "repo_commit": "362af89",
 "alignvid_commit": "9f3e48d76e19fa44e7ca93f0466ca92fb24775c2",
 "protocol_frozen_commit": "ec4d9e4",
 "protocol_frozen_on": "2026-11-18"
}
['Sun Nov 22 04:51:16 2026       ', '+-----------------------------------------------------------------------------------------+', '| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |', '+-----------------------------------------+------------------------+----------------------+', '| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |', '| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |', '|                                         |                        |               MIG M. |', '|=========================================+========================+======================|', '|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |',

In [3]:
# 2. Python 3.11 environment (Kaggle's own Python is 3.13, too new for FramePack's pins); log in /kaggle/working/logs/pip.txt
import os, subprocess, sys
os.makedirs('/kaggle/working/logs', exist_ok=True)
PY = '/kaggle/tmp/venv/bin/python'
with open('/kaggle/working/logs/pip.txt', 'w') as f:
    steps = [[sys.executable, '-m', 'pip', 'install', '-q', 'uv'],
             ['uv', 'venv', '--python', '3.11', '--seed', '/kaggle/tmp/venv'],
             ['uv', 'pip', 'install', '--python', PY, 'torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'],
             ['uv', 'pip', 'install', '--python', PY, '-r', 'gen/requirements-laptop.txt']]
    for s in steps:
        r = subprocess.run(s, stdout=f, stderr=subprocess.STDOUT); print(' '.join(s[:4]), '-> exit', r.returncode); f.flush()
        if r.returncode: break
print(subprocess.run([PY, '-c', 'import torch, transformers, diffusers, numpy, av, matplotlib; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0), transformers.__version__, diffusers.__version__, numpy.__version__)'], capture_output=True, text=True))
subprocess.run(['rm', '-rf', '/root/.cache/uv', '/root/.cache/pip'])


/usr/bin/python3 -m pip install -> exit 0
uv venv --python 3.11 -> exit 0
uv pip install --python -> exit 0
uv pip install --python -> exit 0
CompletedProcess(args=['/kaggle/tmp/venv/bin/python', '-c', 'import torch, transformers, diffusers, numpy, av, matplotlib; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0), transformers.__version__, diffusers.__version__, numpy.__version__)'], returncode=0, stdout='2.11.0+cu128 True Tesla T4 4.46.2 0.33.1 1.26.2\n', stderr='')


CompletedProcess(args=['rm', '-rf', '/root/.cache/uv', '/root/.cache/pip'], returncode=0)

In [4]:
# 3. C1 environment check (GPU, bf16 matmul, versions) -> logs/env_kaggle.*
!/kaggle/tmp/venv/bin/python gen/run_pilot.py --stage env --sync-dir /kaggle/working


[04:52:22] run_pilot start root=/kaggle/tmp/pilot/pilot_package dry=False version={'packaged': '2026-11-22T09:43:36', 'repo_commit': '362af89', 'alignvid_commit': '9f3e48d76e19fa44e7ca93f0466ca92fb24775c2', 'protocol_frozen_commit': 'ec4d9e4', 'protocol_frozen_on': '2026-11-18'}
[04:52:22] --- stage env ---
[04:52:32] env check 2026-11-22T04:52:22 host=kaggle
[04:52:32] platform: Linux-6.18.55+-x86_64-with-glibc2.39
[04:52:32] RAM: 31.35 GB
[04:52:32] free disk: 2507.1 GB
[04:52:32] GPU: {"nvidia_smi": [{"name": "Tesla T4", "vram_total_mb": "15360", "driver": "580.178.04"}, {"name": "Tesla T4", "vram_total_mb": "15360", "driver": "580.178.04"}], "torch_cuda": {"available": true, "version": "12.8", "devices": ["Tesla T4", "Tesla T4"], "total_mem_gb": [14.56, 14.56]}}
[04:52:32] versions: {"python": "3.11.18", "torch": "2.11.0+cu128", "torchvision": "0.26.0+cu128", "diffusers": "0.33.1", "transformers": "4.46.2", "accelerate": "1.6.0", "cv2": "4.10.0", "numpy": "1.26.2", "PIL": "11.1.0",

In [5]:
# 4. source patches (recorded in logs/patches.json, see gen/patches/README.md) and weight download (~40 GB, resumable)
!/kaggle/tmp/venv/bin/python gen/run_pilot.py --stage patch --sync-dir /kaggle/working
!cat logs/patches.json | /kaggle/tmp/venv/bin/python -c "import json,sys; [print(r['id'], 'applied' if r['applied'] else 'already' if r['already'] else r.get('error')) for r in json.load(sys.stdin)]"
!/kaggle/tmp/venv/bin/python gen/run_pilot.py --stage weights --sync-dir /kaggle/working
!du -sh external/AlignVid/models/FramePack/hf_download; df -h /kaggle/tmp | tail -1


[04:52:33] run_pilot start root=/kaggle/tmp/pilot/pilot_package dry=False version={'packaged': '2026-11-22T09:43:36', 'repo_commit': '362af89', 'alignvid_commit': '9f3e48d76e19fa44e7ca93f0466ca92fb24775c2', 'protocol_frozen_commit': 'ec4d9e4', 'protocol_frozen_on': '2026-11-18'}
[04:52:33] --- stage patch ---
[04:52:33] patch dist-backend: applied=True already=False 
[04:52:33] patch lowvram-text-encode-1: applied=True already=False 
[04:52:33] patch lowvram-text-encode-2: applied=True already=False 
[04:52:33] patch sdpa-query-chunks: applied=True already=False 
[04:52:33] patch transformer-dtype-env: applied=True already=False 
[04:52:33] patch fp32-residual: applied=True already=False 
[04:52:33] all stages finished
dist-backend applied
lowvram-text-encode-1 applied
lowvram-text-encode-2 applied
sdpa-query-chunks applied
transformer-dtype-env applied
fp32-residual applied
[04:52:34] run_pilot start root=/kaggle/tmp/pilot/pilot_package dry=False version={'packaged': '2026-11-22T09:43

In [6]:
# 5. C3 dry run (only if DO_DRYRUN): vanilla sample_240, 2 s, --gpu_mem ladder 6 → 8 GB
import subprocess
if DO_DRYRUN:
    subprocess.run(['/kaggle/tmp/venv/bin/python', 'gen/run_pilot.py', '--stage', 'dryrun', '--quiet', '--sync-dir', '/kaggle/working'])
    print(open('logs/run_pilot.log').read()[-2000:])
else:
    print('dry run skipped (DO_DRYRUN = False); gpu_mem comes from the restored run_state.json or the ladder default')


dry run skipped (DO_DRYRUN = False); gpu_mem comes from the restored run_state.json or the ladder default


In [7]:
# 6. C4: length probe (6.0 → 3.0 → 2.0 s, vanilla sample_240) then the arms selected in cell 0, one process per video; results zip in /kaggle/working
import subprocess, json, os
PY = '/kaggle/tmp/venv/bin/python'
if not os.path.exists('logs/run_state.json') or 'gpu_mem' not in json.load(open('logs/run_state.json')):
    json.dump({'gpu_mem': '6'}, open('logs/run_state.json', 'w'))   # the value the C3 dry runs completed with (fp16: GPU peak 13.0 GB)
subprocess.run([PY, 'gen/run_pilot.py', '--stage', 'length', '--quiet', '--sync-dir', '/kaggle/working'])
subprocess.run([PY, 'gen/run_pilot.py', '--stage', 'arms', '--arms', *ARMS, '--quiet', '--sync-dir', '/kaggle/working'])
subprocess.run([PY, 'gen/run_pilot.py', '--stage', 'pack', '--sync-dir', '/kaggle/working'])
print(subprocess.run(['grep', '-E', 'chosen|done in|FAILED|start length|packed|stage', 'logs/run_pilot.log'], capture_output=True, text=True).stdout)
print(open('logs/run_state.json').read())
print(subprocess.run('ls -la /kaggle/working /kaggle/working/outputs/* 2>/dev/null | head -60', shell=True, capture_output=True, text=True).stdout)


[04:56:24] run_pilot start root=/kaggle/tmp/pilot/pilot_package dry=False version={'packaged': '2026-11-22T09:43:36', 'repo_commit': '362af89', 'alignvid_commit': '9f3e48d76e19fa44e7ca93f0466ca92fb24775c2', 'protocol_frozen_commit': 'ec4d9e4', 'protocol_frozen_on': '2026-11-18'}
[04:56:24] --- stage length ---
[04:56:24] vanilla/sample_240: start length=6.0 gpu_mem=6 control_scale=1.0
[sysinfo] /kaggle/tmp/pilot/pilot_package/logs/vanilla_sample_240__len6.0_gm6.json: exit 0, 3561.76 s, child peak RSS 28.52 GB, GPU peak 13037.0 MB
[05:55:54] vanilla/sample_240: done in 3561.76 s, GPU peak 13037.0 MB, RSS 28.52 GB
[05:55:54] length chosen: 6.0 s
[05:55:54] all stages finished
[05:55:55] run_pilot start root=/kaggle/tmp/pilot/pilot_package dry=False version={'packaged': '2026-11-22T09:43:36', 'repo_commit': '362af89', 'alignvid_commit': '9f3e48d76e19fa44e7ca93f0466ca92fb24775c2', 'protocol_frozen_commit': 'ec4d9e4', 'protocol_frozen_on': '2026-11-18'}
[05:55:55] --- stage arms ---
[05:55:

After a session finishes, download `/kaggle/working/results_kaggle_*.zip` (or the whole output) and file it in the repository under `outputs/<arm>/` and `logs/kaggle/`. Every attempt, including failures, has a JSON under `logs/` with wall time, peak GPU memory (nvidia-smi), peak RSS, the transformer dtype and the output tail.
